# 04 — Model Comparison & Chemical Interpretation

ECFP-MLP vs GraphConv 성능 비교, 레이더 차트, False Negative 분자 분석, 작용기별 독성 해석.

In [ ]:
# === Colab setup (run once, restart runtime if prompted) ===
import os, sys
if not os.path.exists("tox21-toxicity-classifier"):
    !git clone https://github.com/zqvo04/tox21-toxicity-classifier.git
%cd tox21-toxicity-classifier
!bash setup_colab.sh
# Make src importable
sys.path.insert(0, os.getcwd())

In [ ]:
# Mount Google Drive for checkpoints (edit path to your own folder)
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/tox21/'
os.makedirs(CKPT_DIR, exist_ok=True)

In [ ]:
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt, seaborn as sns
sns.set_style('whitegrid')
from src import evaluate
FIG = 'results/figures'; os.makedirs(FIG, exist_ok=True)

## 1. 저장된 예측 로딩
노트북 02, 03 에서 저장한 `*_preds.npz` 사용.

In [ ]:
def load_preds(prefix):
    d = np.load(f'{CKPT_DIR}/{prefix}_preds.npz', allow_pickle=True)
    return d['probs'], d['y_true'], d['mask'], list(d['tasks'])

ecfp_probs, y_true, mask, tasks = load_preds('ecfp_mlp')
gcn_probs, y_true_g, mask_g, _   = load_preds('graphconv')

df_ecfp, sum_ecfp = evaluate.evaluate(y_true,   ecfp_probs, mask,   tasks, name='ECFP-MLP')
df_gcn,  sum_gcn  = evaluate.evaluate(y_true_g, gcn_probs,  mask_g, tasks, name='GraphConv')
print(sum_ecfp); print(sum_gcn)

## 2. 성능 비교 테이블

In [ ]:
comp = pd.DataFrame({
    'task': tasks,
    'ECFP_ROC': df_ecfp['roc_auc'].values,
    'GCN_ROC':  df_gcn['roc_auc'].values,
    'ECFP_PR':  df_ecfp['pr_auc'].values,
    'GCN_PR':   df_gcn['pr_auc'].values,
})
comp.loc['MEAN'] = ['MEAN', np.nanmean(comp['ECFP_ROC']), np.nanmean(comp['GCN_ROC']),
                    np.nanmean(comp['ECFP_PR']), np.nanmean(comp['GCN_PR'])]
comp.to_csv(f'{FIG}/comparison_table.csv', index=False)
display(comp.round(4))

## 3. 레이더 차트 (per-task ROC-AUC)

In [ ]:
labels = tasks
e = np.nan_to_num(df_ecfp['roc_auc'].values, nan=0.5)
g = np.nan_to_num(df_gcn['roc_auc'].values, nan=0.5)
angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist()
e = np.concatenate([e, [e[0]]]); g = np.concatenate([g, [g[0]]])
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
ax.plot(angles, e, 'o-', lw=2, label='ECFP-MLP'); ax.fill(angles, e, alpha=0.15)
ax.plot(angles, g, 'o-', lw=2, label='GraphConv'); ax.fill(angles, g, alpha=0.15)
ax.set_thetagrids(np.degrees(angles[:-1]), labels)
ax.set_ylim(0.5, 1.0); ax.set_title('Per-task ROC-AUC', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.2, 1.1))
plt.tight_layout(); plt.savefig(f'{FIG}/radar_comparison.png', dpi=150); plt.show()

In [ ]:
# Bar comparison
x = np.arange(len(tasks)); w = 0.38
plt.figure(figsize=(14, 5))
plt.bar(x - w/2, np.nan_to_num(df_ecfp['roc_auc'],nan=0), w, label='ECFP-MLP')
plt.bar(x + w/2, np.nan_to_num(df_gcn['roc_auc'],nan=0),  w, label='GraphConv')
plt.xticks(x, tasks, rotation=45, ha='right'); plt.ylabel('ROC-AUC'); plt.ylim(0.5,1.0)
plt.legend(); plt.title('ROC-AUC by task'); plt.tight_layout()
plt.savefig(f'{FIG}/roc_auc_bar.png', dpi=150); plt.show()

## 4. False Negative 분자 분석
임상적으로 가장 위험한 오류(독성인데 안전하다고 예측)를 시각화.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
from src.dataset import label_matrix

# Recover SMILES aligned to the test set order via DeepChem scaffold split.
import deepchem as dc
_, (_, _, test_ds), _ = dc.molnet.load_tox21(
    featurizer=dc.feat.CircularFingerprint(radius=2, size=2048),
    splitter='scaffold', reload=True)
test_ids = test_ds.ids

task = 'SR-MMP'; ti = tasks.index(task)
fn_df = evaluate.find_false_negatives(y_true, ecfp_probs, mask, test_ids, ti, threshold=0.5)
print(f'False negatives on {task}:', len(fn_df))
display(fn_df.head(12))

mols, legs = [], []
for _, r in fn_df.head(8).iterrows():
    m = Chem.MolFromSmiles(r['smiles'])
    if m: mols.append(m); legs.append(f"p={r['prob']:.2f}")
img = Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(220,180), legends=legs)
img.save(f'{FIG}/false_negatives_{task}.png'); img

## 5. 작용기별 독성 해석 (Chemical Interpretation)

ECFP는 부분구조(substructure) 기반이라 특정 **작용기(functional group)** 와 독성의
연관을 해석하기 좋다. 대표적 structural alert 들을 RDKit SMARTS로 매칭해
독성 enrichment 를 확인한다.

In [ ]:
from rdkit import Chem
alerts = {
    'Aromatic nitro':      '[$([NX3](=O)=O),$([NX3+](=O)[O-])][c]',
    'Aromatic amine':      'c[NX3;H2,H1]',
    'Michael acceptor':    '[CX3]=[CX3][CX3]=[OX1]',
    'Epoxide':             'C1OC1',
    'Halogenated aromatic':'c[F,Cl,Br,I]',
    'Phenol':              'c[OX2H]',
}
patt = {k: Chem.MolFromSmarts(v) for k, v in alerts.items()}

task = 'SR-MMP'; ti = tasks.index(task)
# Only molecules with a valid (non-missing) label on this task.
valid = mask[:, ti] > 0
smi_valid = test_ids[valid]
lab_valid = y_true[valid, ti]

rows = []
for name, p in patt.items():
    tox_has = tox_no = n_has = n_no = 0
    for smi, lab in zip(smi_valid, lab_valid):
        m = Chem.MolFromSmiles(smi)
        if m is None:
            continue
        if m.HasSubstructMatch(p):
            n_has += 1; tox_has += int(lab == 1)
        else:
            n_no += 1;  tox_no += int(lab == 1)
    r_has = tox_has / max(n_has, 1)
    r_no  = tox_no / max(n_no, 1)
    rows.append(dict(alert=name, n_with=n_has, tox_rate_with=round(r_has, 3),
                     tox_rate_without=round(r_no, 3),
                     enrichment=round(r_has / max(r_no, 1e-6), 2)))
alert_df = pd.DataFrame(rows).sort_values('enrichment', ascending=False)
alert_df.to_csv(f'{FIG}/structural_alerts.csv', index=False)
display(alert_df)

### 화학적 해석 요약

- **Aromatic nitro / amine**: 대사 활성화를 통해 반응성 중간체를 형성 → 미토콘드리아
  독성(SR-MMP), 유전독성과 연관. enrichment 가 높게 나타나는 경향.
- **Michael acceptor / epoxide**: 전자친화성(electrophilic) 으로 단백질·DNA 와 공유결합 →
  세포 스트레스 경로(SR-ARE, SR-p53) 활성화.
- **Halogenated aromatic**: 친유성↑ → 막 투과 및 수용체(NR-AhR) 결합과 관련.

**모델 관점**: ECFP-MLP 는 이런 부분구조 alert 를 직접 비트로 인코딩하므로 해석이 용이하고,
GraphConv 는 원자 간 메시지 전달로 더 유연한 표현을 학습하지만 해석성은 낮다.
데이터가 작고 불균형한 Tox21 에서는 두 접근의 평균 ROC-AUC 가 비슷하게 수렴하는 경우가 많다.

## 6. 최종 결론
비교 테이블·레이더 차트는 `results/figures/` 에 저장되어 README 에서 참조된다.